# News Article Dataset — Exploratory Data Analysis

This notebook queries the SQLite database produced by `ingest.py` and visualizes key properties of the dataset before we begin modeling.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make all plots look clean
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Load the entire cleaned dataset from SQLite
conn = sqlite3.connect('../data/processed/articles.db')
df = pd.read_sql('SELECT * FROM articles', conn)
conn.close()

print(f'Loaded {len(df):,} articles')
df.head(3)

## 1. Dataset Shape & Column Overview

In [ ]:
print('Shape:', df.shape)
print()
print('Dtypes:')
print(df.dtypes)
print()
print('Null counts:')
print(df.isnull().sum())

## 2. Article Count by Publication

In [ ]:
pub_counts = df['publication'].value_counts()

fig, ax = plt.subplots()
pub_counts.plot(kind='barh', ax=ax, color=sns.color_palette('muted', len(pub_counts)))
ax.set_title('Article Count by Publication', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Articles')
ax.invert_yaxis()

# Annotate bars with exact counts
for i, v in enumerate(pub_counts):
    ax.text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print(pub_counts.to_string())

## 3. Article Content Length Distribution

Content length (in characters) tells us whether articles are substantial enough for topic modeling. Very short articles may be noise.

In [ ]:
df['content_len'] = df['content'].str.len()

print('Content length stats (chars):')
print(df['content_len'].describe().apply(lambda x: f'{x:,.0f}'))

fig, axes = plt.subplots(1, 2)

# Full distribution (log scale to handle the long tail)
axes[0].hist(df['content_len'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_yscale('log')
axes[0].set_title('Content Length Distribution (log y-axis)')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Article Count (log)')

# Zoomed to 0–10k chars (covers ~95% of articles)
axes[1].hist(df[df['content_len'] <= 10000]['content_len'], bins=100, color='teal', edgecolor='none')
axes[1].set_title('Content Length — Zoomed to ≤10,000 chars')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Article Count')

plt.tight_layout()
plt.show()

## 4. Very Short Articles — Potential Noise

In [ ]:
# Articles under 200 chars are likely stubs or data errors
short = df[df['content_len'] < 200][['title', 'publication', 'content_len', 'content']]
print(f'Articles under 200 chars: {len(short):,}')
short.head(10)

## 5. Content Length by Publication

Different outlets write very different length articles — this affects TF-IDF feature quality.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

order = df.groupby('publication')['content_len'].median().sort_values(ascending=False).index
sns.boxplot(
    data=df[df['content_len'] <= 15000],  # clip outliers for readability
    x='publication', y='content_len',
    order=order, ax=ax, palette='muted'
)
ax.set_title('Content Length by Publication (articles ≤15k chars)', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Characters')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

## 6. Articles Over Time

In [ ]:
# Parse date and group by month
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
monthly = df.dropna(subset=['date_parsed']).set_index('date_parsed').resample('ME').size()

fig, ax = plt.subplots()
monthly.plot(ax=ax, color='steelblue', linewidth=1.5)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='steelblue')
ax.set_title('Articles Published per Month', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Article Count')
plt.tight_layout()
plt.show()

## 7. Missing Data Summary

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]  # only show columns with missing data

fig, ax = plt.subplots(figsize=(7, 4))
null_pct.plot(kind='bar', ax=ax, color='salmon', edgecolor='none')
ax.set_title('Missing Data by Column (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('% Missing')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(null_pct):
    ax.text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Key Takeaways for Modeling

Fill these in after reviewing the plots above.

In [ ]:
print('=== Dataset Summary ===')
print(f"Total articles:          {len(df):,}")
print(f"Publications:            {df['publication'].nunique()}")
print(f"Date range:              {df['date_parsed'].min().date()} → {df['date_parsed'].max().date()}")
print(f"Median article length:   {df['content_len'].median():,.0f} chars")
print(f"Articles < 200 chars:    {(df['content_len'] < 200).sum():,}  (likely noise — filter in transform.py)")
print(f"Missing author:          {df['author'].isnull().sum():,} ({df['author'].isnull().mean()*100:.1f}%)")
print(f"Missing date:            {df['date'].isnull().sum():,} ({df['date'].isnull().mean()*100:.1f}%)")

## 9. Discovered Topics (from BERTopic)

This section reads directly from `data/processed/topic_info.csv` — the artifact saved by `topic_model.py`. Re-run this cell any time after running the pipeline to see the current topic list.

In [ ]:
from pathlib import Path

topic_info_path = Path('../data/processed/topic_info.csv')

if not topic_info_path.exists():
    print("topic_info.csv not found — run topic_model.py first.")
else:
    ti = pd.read_csv(topic_info_path)
    ti = ti[ti['Topic'] != -1].copy()
    ti['keywords'] = ti['Name'].str.split('_').str[1:].str.join(', ')
    ti = ti[['Topic', 'Count', 'keywords']].sort_values('Topic').reset_index(drop=True)

    total = ti['Count'].sum()
    ti['% of labeled'] = (ti['Count'] / total * 100).round(1)

    print(f"Topics discovered: {len(ti)}")
    print(f"Total labeled articles: {total:,}\n")
    print(ti.to_string(index=False))

    # Bar chart of article distribution across topics
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = sns.color_palette('muted', len(ti))
    ax.barh(ti['keywords'], ti['Count'], color=colors)
    ax.set_title('Articles per Discovered Topic', fontsize=13, fontweight='bold')
    ax.set_xlabel('Number of Articles')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()